In [3]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import pandas as pd
from pydantic import BaseModel, Field
from typing import List

In [2]:
doc = PyPDFLoader('data/fattura.pdf').load() # pip install pypdf

doc

[Document(metadata={'producer': 'GPL Ghostscript 10.05.1', 'creator': 'InvoiceHome.com', 'creationdate': "D:20250723100914Z00'00'", 'moddate': "D:20250723100914Z00'00'", 'author': 'InvoiceHome.com', 'source': 'data/fattura.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Azienda srl\nBill To Invoice #\n42\nAzienda SpA Invoice Date 23/07/2025\nDescription Amount\n1 x DLRG-23 Faretti LED 40W RGB 118.50\n1 x STRD-213 Asta microfonica Rode PSA1+ per microfoni a condensatore 104. 99\n1 x MCRD-12 Microfono Rode PodMic-USB 139.00\n2 x CVSB-71 Cavo USB-C / USB-A 3 metri schermato 9.99\n1 x WBLG-9 Webcam Logitech Brio 4k con supporto da monitor 129.91\n1 x STDK-3 Elgato Stream Deck 15 tasti 119.99\nSubtotal 622.38\ni.v.a. 22.0% 136.93\nInvoice Total 759.31 €\nTerms & Conditions\nIl pagamento deve essere e9ettuato entro 15 giorni')]

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=os.getenv("openai_key"))

In [4]:
print(llm.invoke("elenca articoli e quantità presenti nella seguente fattura:\n" + doc[0].page_content).content)

Ecco l'elenco degli articoli e delle quantità presenti nella fattura:

1. **DLRG-23 Faretti LED 40W RGB** - 1 unità - **118.50 €**
2. **STRD-213 Asta microfonica Rode PSA1+ per microfoni a condensatore** - 1 unità - **104.99 €**
3. **MCRD-12 Microfono Rode PodMic-USB** - 1 unità - **139.00 €**
4. **CVSB-71 Cavo USB-C / USB-A 3 metri schermato** - 2 unità - **9.99 €** ciascuna
5. **WBLG-9 Webcam Logitech Brio 4k con supporto da monitor** - 1 unità - **129.91 €**
6. **STDK-3 Elgato Stream Deck 15 tasti** - 1 unità - **119.99 €**

Se hai bisogno di ulteriori informazioni o chiarimenti, fammi sapere!


In [5]:
class Product(BaseModel):
    """Details of an item or product present in a document"""

    product_code: str = Field(default="--", description="the product code associated with an item")
    description: str = Field(default=None, description="the description associated with an item")
    quantity: str = Field(default=0,
                          description="how many items are in an order, can have different units of measurement")
    price: float = Field(default=0., description="the price of the item")


class Products_List(BaseModel):
    """Identifying information about all items in a document"""

    products: List[Product]

In [6]:
products_llm = llm.with_structured_output(Products_List)

prompt = PromptTemplate(template="elenca codice articoli, descrizione articoli, quantità e prezzo presenti nella seguente fattura:\n{doc}", input_variables=["doc"], )

chain = prompt | products_llm

In [7]:
articoli_fattura = chain.invoke({"doc": doc[0].page_content})

In [8]:
articoli_fattura

Products_List(products=[Product(product_code='DLRG-23', description='Faretti LED 40W RGB', quantity='1', price=118.5), Product(product_code='STRD-213', description='Asta microfonica Rode PSA1+ per microfoni a condensatore', quantity='1', price=104.99), Product(product_code='MCRD-12', description='Microfono Rode PodMic-USB', quantity='1', price=139.0), Product(product_code='CVSB-71', description='Cavo USB-C / USB-A 3 metri schermato', quantity='2', price=9.99), Product(product_code='WBLG-9', description='Webcam Logitech Brio 4k con supporto da monitor', quantity='1', price=129.91), Product(product_code='STDK-3', description='Elgato Stream Deck 15 tasti', quantity='1', price=119.99)])

In [9]:
for articolo in articoli_fattura.products:
    print(articolo)

product_code='DLRG-23' description='Faretti LED 40W RGB' quantity='1' price=118.5
product_code='STRD-213' description='Asta microfonica Rode PSA1+ per microfoni a condensatore' quantity='1' price=104.99
product_code='MCRD-12' description='Microfono Rode PodMic-USB' quantity='1' price=139.0
product_code='CVSB-71' description='Cavo USB-C / USB-A 3 metri schermato' quantity='2' price=9.99
product_code='WBLG-9' description='Webcam Logitech Brio 4k con supporto da monitor' quantity='1' price=129.91
product_code='STDK-3' description='Elgato Stream Deck 15 tasti' quantity='1' price=119.99


In [10]:
df = pd.DataFrame([{
    "Codice Prodotto": p.product_code,
    "Descrizione": p.description,
    "Quantità": p.quantity,
    "Prezzo": p.price
} for p in articoli_fattura.products])

display(df)

,Codice Prodotto,Descrizione,Quantità,Prezzo
0,DLRG-23,Faretti LED 40W RGB,1,118.50
1,STRD-213,Asta microfonica Rode PSA1+ per microfoni a co...,1,104.99
2,MCRD-12,Microfono Rode PodMic-USB,1,139.00
3,CVSB-71,Cavo USB-C / USB-A 3 metri schermato,2,9.99
4,WBLG-9,Webcam Logitech Brio 4k con supporto da monitor,1,129.91
5,STDK-3,Elgato Stream Deck 15 tasti,1,119.99


In [11]:
products_llm.invoke("ho due canarini gialli")

Products_List(products=[Product(product_code='CAN001', description='Canarini gialli, canaries yellow in color, known for their singing abilities.', quantity='2', price=50.0)])